# PolicyPilot: long context vs RAG on a free Colab T4

Runs the pre-registered grid in `docs/LONG_CONTEXT_PLAN.md`: Qwen2.5-7B-Instruct (Q4_K_M, 32K
context) served by Ollama, reading the policy library in context, against single-shot RAG with the
same model. No paid API, no keys.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.

**Free Colab disconnects** (idle tabs, or after a few hours). Every finished requirement is saved to
Google Drive (`MyDrive/policypilot_long_context/long_context_runs.jsonl`), so after a disconnect
reconnect and **Run all** again: finished rows are skipped. Expect several sessions for the whole
grid (about 1,000 model calls).

In [ ]:
# 1. Save progress to Google Drive, so a disconnect loses nothing
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
OUT = "/content/drive/MyDrive/policypilot_long_context"
os.makedirs(OUT, exist_ok=True)
RUNS = f"{OUT}/long_context_runs.jsonl"
print("Checkpoint:", RUNS, "rows so far:", sum(1 for _ in open(RUNS)) if os.path.exists(RUNS) else 0)

In [ ]:
# 2. Get the code
if not os.path.exists("/content/policypilot"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "long-context",
                    "https://github.com/vik-bit/policypilot.git", "/content/policypilot"], check=True)
os.chdir("/content/policypilot")
!git log -1 --oneline
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Install PolicyPilot (without its web/agent extras, which long-context mode does not import)
#    and Ollama (official installer). Works on any Python version.
!pip install -q --no-deps -e .
!pip install -q "fastembed>=0.4" "tokenizers>=0.20" pypdf
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 4. Start Ollama, download Qwen2.5-7B-Instruct (4.7 GB) and give it a 32K context.
#    Ollama's default context is much smaller and it silently drops the start of longer prompts,
#    so the eval also checks once that the server read the whole largest prompt.
import time, urllib.request
env = {**os.environ, "OLLAMA_NUM_PARALLEL": "1", "OLLAMA_KEEP_ALIVE": "-1", "OLLAMA_MAX_LOADED_MODELS": "1"}
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=5); break
    except OSError:
        time.sleep(2)
!ollama pull qwen2.5:7b-instruct
!echo "FROM qwen2.5:7b-instruct" > Modelfile
!echo "PARAMETER num_ctx 32768" >> Modelfile
!ollama create qwen2.5:7b-instruct-32k -f Modelfile
MODEL, BASE = "qwen2.5:7b-instruct-32k", "http://localhost:11434/v1"
# If the smoke check below fails with HTTP 400 about response_format, change this to "json_object".
STRUCTURED = "json_schema"

In [ ]:
# 5. Exact prompt sizes with the Qwen tokenizer (no model call)
!python -m policypilot.longctx tokens --distractors 0 50 133

In [ ]:
# 6. Smoke check on 5 requirements of the development set (not the held-out set)
!python -m policypilot.longctx --base-url {BASE} --model {MODEL} --structured {STRUCTURED} eval --grid smoke --out /content/smoke_runs.jsonl
!tail -n 2 /content/smoke_runs.jsonl | cut -c 1-600
!nvidia-smi --query-gpu=memory.used --format=csv

In [ ]:
# 7. The pre-registered grid on the held-out `real` set. Safe to re-run: finished rows are skipped.
!python -m policypilot.longctx --base-url {BASE} --model {MODEL} --structured {STRUCTURED} eval --grid main --out "{RUNS}"

In [ ]:
# 8. Tables, intervals and paired tests (no model call)
import shutil
shutil.copy(RUNS, "data/eval/long_context_runs.jsonl")
!python -m policypilot.longctx report
shutil.copy("docs/LONG_CONTEXT_RESULTS.md", f"{OUT}/LONG_CONTEXT_RESULTS.md")
shutil.copy("data/eval/long_context_results.json", f"{OUT}/long_context_results.json")